In [ ]:
import pandas as pd
from itertools import combinations
from pathlib import Path
import json

In [6]:
DATA_DIR = Path("task3_outputs")
OUTPUT_DIR = Path("task6_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

category_cooccurrence = pd.read_csv(DATA_DIR / "category_cooccurrence.csv", index_col=0)

# the final 10 from Task #5 (Maya's Step 9) -- edit this if the team's final list differs
final_10 = [
    "Governing Law",
    "Renewal Term",
    "Revenue/Profit Sharing",
    "Cap On Liability",
    "Uncapped Liability",
    "Termination For Convenience",
    "Anti-Assignment",
    "Audit Rights",
    "License Grant",
    "Exclusivity",
]

assert len(final_10) == 10, f"expected 10 categories, got {len(final_10)}"
print(f"{len(final_10)} categories loaded")

FileNotFoundError: [Errno 2] No such file or directory: 'task3_outputs\\category_cooccurrence.csv'

In [7]:
# case-insensitive matching against the matrix's row/column labels
matrix_names = {name.strip().lower(): name for name in category_cooccurrence.index}
resolved = {}
for name in final_10:
    key = name.strip().lower()
    if key in matrix_names:
        resolved[name] = matrix_names[key]
    else:
        print(f"WARNING -- '{name}' not found in category_cooccurrence.csv -- check spelling")

pair_rows = []
for a, b in combinations(final_10, 2):
    if a in resolved and b in resolved:
        count = category_cooccurrence.loc[resolved[a], resolved[b]]
        pair_rows.append({"category_a": a, "category_b": b, "cooccurrence_count": int(count)})

pair_strength = pd.DataFrame(pair_rows).sort_values("cooccurrence_count", ascending=False).reset_index(drop=True)
pair_strength.to_csv(OUTPUT_DIR / "pair_strength_all_45.csv", index=False)
pair_strength

NameError: name 'category_cooccurrence' is not defined

In [8]:
proposed_bundles = [
    {
        "bundle": 1, "category_a": "Cap On Liability", "category_b": "Uncapped Liability",
        "rationale": "Mutually exclusive liability structures -- a reviewer needs to know which one applies, so reviewing them together directly answers 'what's our risk exposure here?'",
        "alternative_considered": "Cap On Liability + Audit Rights",
        "alternative_reason": "Considered pairing liability exposure with the mechanism to monitor for breaches, but the direct cap-vs-uncapped contrast was judged more immediately useful to a reviewer.",
    },
    {
        "bundle": 2, "category_a": "License Grant", "category_b": "Exclusivity",
        "rationale": "Both define the scope of what's being licensed -- License Grant states what rights exist, Exclusivity states whether anyone else can also have them.",
        "alternative_considered": "License Grant + Anti-Assignment",
        "alternative_reason": "Considered pairing on 'what rights exist and whether they can move,' but Exclusivity ties more directly to the license's actual scope.",
    },
    {
        "bundle": 3, "category_a": "Renewal Term", "category_b": "Termination For Convenience",
        "rationale": "Contract lifecycle bookends -- how the agreement continues vs. how either party can end it early.",
        "alternative_considered": "Renewal Term + Governing Law",
        "alternative_reason": "Both are 'structural' clauses, but pairing renewal with termination is more directly actionable for a reviewer assessing contract flexibility.",
    },
    {
        "bundle": 4, "category_a": "Revenue/Profit Sharing", "category_b": "Audit Rights",
        "rationale": "Audit Rights are often the specific mechanism used to verify compliance with revenue/profit-sharing terms -- reviewing them together checks both the obligation and how it's enforced.",
        "alternative_considered": "Revenue/Profit Sharing + Exclusivity",
        "alternative_reason": "Exclusivity deals can have revenue implications, but Audit Rights connects more mechanically to verifying the revenue terms themselves.",
    },
    {
        "bundle": 5, "category_a": "Governing Law", "category_b": "Anti-Assignment",
        "rationale": "Both are structural/enforceability terms -- which jurisdiction's law applies, and whether contractual rights can be transferred under it.",
        "alternative_considered": "Governing Law + Renewal Term",
        "alternative_reason": "Both are administrative clauses, but Anti-Assignment pairs more naturally with Governing Law's enforceability angle than with renewal timing.",
    },
]

bundles_df = pd.DataFrame(proposed_bundles)

# attach the real co-occurrence count for each chosen pair, for reference
bundles_df = bundles_df.merge(
    pair_strength[["category_a", "category_b", "cooccurrence_count"]],
    on=["category_a", "category_b"], how="left"
)

bundles_df.to_csv(OUTPUT_DIR / "proposed_bundles.csv", index=False)
bundles_df

NameError: name 'pair_strength' is not defined

In [9]:
errors = []

if len(bundles_df) != 5:
    errors.append(f"expected 5 bundles, found {len(bundles_df)}")

all_categories_used = list(bundles_df["category_a"]) + list(bundles_df["category_b"])
if len(all_categories_used) != len(set(all_categories_used)):
    dupes = [c for c in set(all_categories_used) if all_categories_used.count(c) > 1]
    errors.append(f"category used in more than one bundle: {dupes}")

missing = set(final_10) - set(all_categories_used)
if missing:
    errors.append(f"category from the final 10 never appears in a bundle: {missing}")

extra = set(all_categories_used) - set(final_10)
if extra:
    errors.append(f"category in a bundle that isn't in the final 10: {extra}")

if errors:
    print("FAILED:")
    for e in errors:
        print(f"  - {e}")
else:
    print("PASSED -- 5 bundles, 2 categories each, all 10 categories covered exactly once.")

NameError: name 'final_10' is not defined

In [ ]:
task6_summary = {
    "num_bundles": len(bundles_df),
    "categories_covered": sorted(all_categories_used),
    "checks_passed": len(errors) == 0,
}

with open(OUTPUT_DIR / "task6_summary.json", "w") as f:
    json.dump(task6_summary, f, indent=2)

task6_summary